# 🏭 Defect Detection in Manufacturing Using AI

This notebook builds an AI-powered defect detection system using computer vision and machine learning.

---

## 📦 Dataset Sources

Use one of the following datasets:

| Dataset | Link | Description |
|--------|------|-------------|
| **MVTec AD** (Recommended) | [Kaggle Search](https://www.kaggle.com/datasets/search?q=mvtec+defect) | Industrial defect dataset with 15 categories |
| **NEU Steel Surface Defect** | [Kaggle](https://www.kaggle.com/datasets/kaustubhdikshit/neu-surface-defect-database) | 6 types of steel surface defects |
| **PCB Defect Dataset** | [Kaggle Search](https://www.kaggle.com/datasets/search?q=PCB+defect+detection) | Circuit board defects |
| **Casting Product Defect** | [Kaggle](https://www.kaggle.com/datasets/ravirajsinh45/real-life-industrial-dataset-of-casting-product) | Real industrial casting defects |

### ⬇️ Download Instructions (Kaggle API)
```bash
pip install kaggle
# Place your kaggle.json API key in ~/.kaggle/
kaggle datasets download -d ravirajsinh45/real-life-industrial-dataset-of-casting-product
unzip real-life-industrial-dataset-of-casting-product.zip -d casting_data
```

## 1. Install & Import Libraries

In [ ]:
# Install required packages (run once)
!pip install tensorflow opencv-python matplotlib scikit-learn seaborn pillow tqdm

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Sklearn
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

## 2. Configuration & Hyperparameters

In [ ]:
# ===================== CONFIG =====================
DATA_DIR = 'casting_data'        # <-- Change to your dataset path
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-4
NUM_CLASSES = 2                  # Adjust based on your dataset
CLASS_NAMES = ['ok_front', 'def_front']  # Adjust to match your folder names
SEED = 42
# ==================================================

## 3. Data Loading & Exploration

In [ ]:
def load_image_paths_and_labels(data_dir, class_names):
    """Loads image paths and labels from a directory."""
    paths, labels = [], []
    for label, cls in enumerate(class_names):
        cls_dir = os.path.join(data_dir, cls)
        if not os.path.exists(cls_dir):
            print(f"Warning: Directory not found: {cls_dir}")
            continue
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                paths.append(os.path.join(cls_dir, fname))
                labels.append(label)
    return paths, labels

image_paths, image_labels = load_image_paths_and_labels(DATA_DIR, CLASS_NAMES)
print(f"Total images found: {len(image_paths)}")
print(f"Class distribution: {pd.Series(image_labels).value_counts().to_dict()}")

In [ ]:
# Visualize class distribution
counts = pd.Series(image_labels).value_counts()
plt.figure(figsize=(7, 4))
sns.barplot(x=[CLASS_NAMES[i] for i in counts.index], y=counts.values, palette='viridis')
plt.title('Class Distribution')
plt.ylabel('Number of Images')
plt.xlabel('Class')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('Sample Images Per Class', fontsize=14)
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    cls_images = [p for p, l in zip(image_paths, image_labels) if l == cls_idx]
    for j in range(min(5, len(cls_images))):
        img = cv2.imread(cls_images[j])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[cls_idx][j].imshow(img)
        axes[cls_idx][j].set_title(cls_name, fontsize=9)
        axes[cls_idx][j].axis('off')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing & Augmentation

In [ ]:
# Train/Val/Test Split
X_train_p, X_test_p, y_train, y_test = train_test_split(
    image_paths, image_labels, test_size=0.2, random_state=SEED, stratify=image_labels)
X_train_p, X_val_p, y_train, y_val = train_test_split(
    X_train_p, y_train, test_size=0.15, random_state=SEED, stratify=y_train)

print(f"Train: {len(X_train_p)} | Val: {len(X_val_p)} | Test: {len(X_test_p)}")

In [ ]:
def preprocess_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

def augment_image(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.image.rot90(img, k=tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32))
    return img, label

# Create tf.data datasets
def make_dataset(paths, labels, augment=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED)
    ds = ds.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(X_train_p, y_train, augment=True, shuffle=True)
val_ds   = make_dataset(X_val_p,   y_val)
test_ds  = make_dataset(X_test_p,  y_test)

print("Datasets created successfully.")

## 5. Model Building — Transfer Learning with MobileNetV2

In [ ]:
def build_model(num_classes, img_size=IMG_SIZE, learning_rate=LEARNING_RATE):
    base_model = MobileNetV2(
        input_shape=(*img_size, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False  # Freeze base model initially

    inputs = keras.Input(shape=(*img_size, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    if num_classes == 2:
        outputs = layers.Dense(1, activation='sigmoid')(x)
        loss = 'binary_crossentropy'
        metrics = ['accuracy', tf.keras.metrics.AUC(name='auc')]
    else:
        outputs = layers.Dense(num_classes, activation='softmax')(x)
        loss = 'sparse_categorical_crossentropy'
        metrics = ['accuracy']

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=loss,
        metrics=metrics
    )
    return model

model = build_model(NUM_CLASSES)
model.summary()

## 6. Training Phase 1 — Train Top Layers

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1)
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

## 7. Fine-Tuning — Unfreeze Top Layers of Base Model

In [ ]:
# Unfreeze last 30 layers of MobileNetV2
base_model = model.layers[1]  # MobileNetV2
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE / 10),
    loss=model.loss,
    metrics=model.metrics_names[1:] if hasattr(model, 'metrics_names') else ['accuracy']
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("Fine-tuning complete!")

## 8. Training History Visualization

In [ ]:
def plot_history(histories, labels):
    """Combine and plot training histories."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for hist, label in zip(histories, labels):
        offset = 0
        epochs = range(1, len(hist.history['loss']) + 1)
        axes[0].plot(epochs, hist.history['accuracy'], label=f'{label} Train Acc')
        axes[0].plot(epochs, hist.history['val_accuracy'], linestyle='--', label=f'{label} Val Acc')
        axes[1].plot(epochs, hist.history['loss'], label=f'{label} Train Loss')
        axes[1].plot(epochs, hist.history['val_loss'], linestyle='--', label=f'{label} Val Loss')
    
    axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].legend(fontsize=8)
    axes[1].set_title('Loss');     axes[1].set_xlabel('Epoch'); axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

plot_history([history1, history2], ['Phase1', 'Phase2'])

## 9. Evaluation on Test Set

In [ ]:
# Load best model
model = keras.models.load_model('best_model.h5')

test_results = model.evaluate(test_ds, verbose=1)
print("\n📊 Test Results:")
for name, val in zip(model.metrics_names, test_results):
    print(f"  {name}: {val:.4f}")

In [ ]:
# Predictions
y_pred_prob = model.predict(test_ds, verbose=1)
if NUM_CLASSES == 2:
    y_pred = (y_pred_prob.flatten() > 0.5).astype(int)
else:
    y_pred = np.argmax(y_pred_prob, axis=1)

print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 10. Grad-CAM Visualization (Explainability)

In [ ]:

import tensorflow as tf

# Find last conv layer inside MobileNetV2
base_model = model.get_layer("mobilenetv2_1.00_224")
last_conv_layer = "out_relu"

def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[
            base_model.get_layer(last_conv_layer_name).output,
            model.output
        ]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)

        if predictions.shape[-1] == 1:
            class_channel = predictions[:, 0]
        else:
            pred_index = tf.argmax(predictions[0])
            class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)

    return heatmap.numpy()

def display_gradcam(img_source, model):

    # img_source may be a file path OR an image array
    if isinstance(img_source, str):
        img = tf.io.read_file(img_source)
        img_tensor = tf.image.decode_jpeg(img, channels=3)
    else:
        img_tensor = tf.convert_to_tensor(img_source)

        if len(img_tensor.shape) == 2:
            img_tensor = tf.stack([img_tensor]*3, axis=-1)

    img_resized = tf.cast(
        tf.image.resize(img_tensor, IMG_SIZE),
        tf.float32
    ) / 255.0

    img_array = tf.expand_dims(img_resized, axis=0)

    heatmap = make_gradcam_heatmap(
        img_array,
        model,
        last_conv_layer
    )

    orig_img = tf.image.resize(img_tensor, IMG_SIZE).numpy().astype("uint8")

    import cv2
    heatmap = cv2.resize(heatmap, IMG_SIZE)
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    overlay = cv2.addWeighted(orig_img, 0.6, heatmap, 0.4, 0)

    plt.figure(figsize=(10,5))

    plt.subplot(1,2,1)
    plt.imshow(orig_img)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    plt.title("Grad-CAM")
    plt.axis("off")

    plt.show()

defective_samples = [p for p, l in zip(X_test_p, y_test) if l == 1]

if defective_samples:
    display_gradcam(defective_samples[0], model)
else:
    print("No defective samples found.")


## 11. Real-Time Defect Prediction on New Images

In [ ]:
def predict_single_image(img_path, model, class_names=CLASS_NAMES):
    """Predict defect on a single image and display result."""
    img = Image.open(img_path).convert('RGB').resize(IMG_SIZE)
    img_array = np.array(img) / 255.0
    img_tensor = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_tensor, verbose=0)
    if NUM_CLASSES == 2:
        prob = float(prediction[0][0])
        pred_class = 1 if prob > 0.5 else 0
        confidence = prob if pred_class == 1 else 1 - prob
    else:
        pred_class = np.argmax(prediction[0])
        confidence = prediction[0][pred_class]

    label = class_names[pred_class]
    color = 'red' if 'def' in label.lower() else 'green'

    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.title(f"Prediction: {label}\nConfidence: {confidence:.2%}",
              color=color, fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    return label, confidence

# Test on a sample image
if X_test_p:
    label, conf = predict_single_image(X_test_p[0], model)
    print(f"Result: {label} ({conf:.2%} confidence)")

## 12. Data Logging & Defect Report

In [ ]:
def generate_defect_report(image_paths_list, model, class_names=CLASS_NAMES):
    """Batch predict and log results to a DataFrame."""
    records = []
    for path in tqdm(image_paths_list, desc="Inspecting"):
        img = Image.open(path).convert('RGB').resize(IMG_SIZE)
        img_array = np.expand_dims(np.array(img) / 255.0, axis=0)
        pred = model.predict(img_array, verbose=0)
        if NUM_CLASSES == 2:
            prob = float(pred[0][0])
            cls = 1 if prob > 0.5 else 0
            conf = prob if cls == 1 else 1 - prob
        else:
            cls = int(np.argmax(pred[0]))
            conf = float(pred[0][cls])
        records.append({
            'image': os.path.basename(path),
            'prediction': class_names[cls],
            'confidence': round(conf, 4),
            'is_defective': 'YES' if 'def' in class_names[cls].lower() else 'NO',
            'severity': 'HIGH' if conf > 0.9 else ('MEDIUM' if conf > 0.7 else 'LOW')
        })
    return pd.DataFrame(records)

# Run on test images (first 50 for demo)
report_df = generate_defect_report(X_test_p[:50], model)
print(report_df.head(10))
report_df.to_csv('defect_report.csv', index=False)
print("\n✅ Report saved to defect_report.csv")

In [ ]:
# Summary statistics
print("\n📊 Defect Summary:")
print(report_df['is_defective'].value_counts())
print("\n🔴 Severity Breakdown:")
print(report_df[report_df['is_defective'] == 'YES']['severity'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
report_df['is_defective'].value_counts().plot.pie(ax=axes[0], autopct='%1.1f%%',
    colors=['#2ecc71','#e74c3c'], title='Defective vs OK')
report_df[report_df['is_defective']=='YES']['severity'].value_counts().plot.bar(
    ax=axes[1], color=['#e74c3c','#f39c12','#3498db'], title='Defect Severity')
plt.tight_layout()
plt.show()

## 13. Save the Final Model

In [ ]:
# Save in SavedModel format
model.save('defect_detection_model')
print("✅ Model saved as 'defect_detection_model' directory")

# Also save as .h5
model.save('defect_detection_model.h5')
print("✅ Model saved as 'defect_detection_model.h5'")

---

## ✅ Project Summary

| Component | Details |
|-----------|--------|
| **Model** | MobileNetV2 (Transfer Learning + Fine-tuning) |
| **Input** | Product images (224×224 RGB) |
| **Output** | Defect class + Confidence score + Severity |
| **Key Features** | Automated detection, Grad-CAM explainability, batch logging |
| **Applicable Industries** | Automobile, Electronics, Pharmaceuticals, Steel |

### 🚀 Next Steps
- Deploy as a REST API using **FastAPI** or **Flask**
- Integrate with a live camera feed using **OpenCV VideoCapture**
- Try **YOLOv8** for real-time object-level defect localization
- Use **MLflow** for experiment tracking